<a href="https://colab.research.google.com/github/divikverma/HelloWorld/blob/master/PatientRiskStratification.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Step 1: Setup, Data Loading, and Integration
What we are doing: We are importing our necessary Python libraries, loading the structured patient data and the unstructured encounter data, and merging them together based on the patient_id.
Why: To predict a patient's risk, an AI model needs a holistic view. By joining these datasets, we combine demographic/lifestyle risk factors (Age, BMI, Smoking) with clinical history (Diagnosis codes and Doctor's notes).

In [10]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# 1. Load the datasets
# (Ensure the CSV files are in the same directory as your notebook, or update the paths)
patients_df = pd.read_csv('healthcare_patients.csv')
encounters_df = pd.read_csv('healthcare_encounters.csv')

print(f"Loaded {len(patients_df)} patients and {len(encounters_df)} encounters.")

# 2. Merge the datasets on 'patient_id'
# We use a right join (or inner) so we only keep encounters that have patient details
df_merged = pd.merge(encounters_df, patients_df, on='patient_id', how='left')

# Sort by patient and date to see the chronological history
df_merged = df_merged.sort_values(by=['patient_id', 'date']).reset_index(drop=True)

print("Merged Dataset Preview:")
display(df_merged.head())

Loaded 400 patients and 1500 encounters.
Merged Dataset Preview:


,encounter_id,patient_id,date,dx_code,note,age,sex,bmi,smoker,diabetic
0,V00485,H10000,2024-03-15,E78,Patient presents with headache for 1 days. His...,68,M,25.3,0,0
1,V01169,H10001,2024-02-28,I25,Patient presents with anxiety for 10 days. His...,70,M,23.4,1,0
2,V01107,H10001,2024-03-12,I25,Patient presents with headache for 1 days. His...,70,M,23.4,1,0
3,V00117,H10001,2024-03-27,E11,Follow-up visit. hypertension controlled. Reco...,70,M,23.4,1,0
4,V00048,H10001,2024-04-06,E11,Patient presents with insomnia for 7 days. His...,70,M,23.4,1,0


Step 2: Creating a Structured "Baseline Risk" Score
What we are doing: We are calculating a preliminary risk score based purely on the structured data. We assign points if the patient is older, has a high BMI, smokes, or has diabetes.
Why: Before applying complex Generative AI, it is best practice to establish a baseline using standard medical heuristics. This structured risk score will later be combined with the AI's analysis of the unstructured notes to create a final, highly accurate prediction.

In [11]:
def calculate_baseline_risk(row):
    risk_score = 0
    # Age factor: higher risk for older patients
    if row['age'] > 65: risk_score += 2
    elif row['age'] > 50: risk_score += 1

    # BMI factor: Obesity (BMI >= 30) adds risk
    if row['bmi'] >= 30.0: risk_score += 1

    # Lifestyle and chronic conditions
    if row['smoker'] == 1: risk_score += 2
    if row['diabetic'] == 1: risk_score += 2

    return risk_score

# Apply the function to create a new feature column
df_merged['structured_risk_score'] = df_merged.apply(calculate_baseline_risk, axis=1)

print("Patients with highest structured risk:")
display(df_merged[['patient_id', 'age', 'bmi', 'smoker', 'diabetic', 'structured_risk_score']].drop_duplicates().sort_values(by='structured_risk_score', ascending=False).head())

Patients with highest structured risk:


,patient_id,age,bmi,smoker,diabetic,structured_risk_score
185,H10051,89,31.6,1,1,7
36,H10009,67,31.4,1,1,7
388,H10109,67,26.8,1,1,6
119,H10033,74,24.2,1,1,6
670,H10184,51,30.3,1,1,6


Step 3: Generating Embeddings for Clinical Notes
What we are doing: We are using a pre-trained Natural Language Processing (NLP) model to read the note column (e.g., "Patient presents with severe chest pain...") and convert it into high-dimensional numerical vectors (embeddings).
Why: Machine learning algorithms cannot do math on raw text. By converting notes into embeddings, the AI captures the semantic meaning and severity of the text. Notes describing mild symptoms will end up mathematically "far away" from notes describing life-threatening symptoms, allowing our risk model to learn the difference.

Note: You will need to install the sentence-transformers library if you haven't already (!pip install sentence-transformers).

In [12]:
# !pip install sentence-transformers
from sentence_transformers import SentenceTransformer

# Load a lightweight, fast pre-trained embedding model
# 'all-MiniLM-L6-v2' is excellent for general text similarity and embedding tasks
print("Loading embedding model...")
embedder = SentenceTransformer('all-MiniLM-L6-v2')

# Extract the clinical notes as a list
notes = df_merged['note'].fillna("No clinical notes available").tolist()

# Generate embeddings for all notes
print("Generating embeddings for clinical notes. This may take a moment...")
note_embeddings = embedder.encode(notes, show_progress_bar=True)

# Add the embeddings back to our dataframe for later use
df_merged['note_embedding'] = list(note_embeddings)

print(f"Successfully generated embeddings of shape: {note_embeddings.shape}")
# This means every note is now represented by a vector of 384 numbers!

Loading embedding model...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Generating embeddings for clinical notes. This may take a moment...


Batches:   0%|          | 0/47 [00:00<?, ?it/s]

Successfully generated embeddings of shape: (1500, 384)


Step 4: Setting up the RAG (Retrieval-Augmented Generation) Framework
What we are doing: We are simulating a vector database. We write a function that takes a specific patient_id and retrieves all of their historical clinical notes.
Why: This is the core of the Generative AI value-add. If our model flags a patient as "High Risk," a doctor won't just trust a black-box number. They need context. We will retrieve these specific notes and pass them to an LLM (like GPT-4, Claude, or Gemini) to generate a summary saying: "I flagged this patient as high risk because their last 3 notes mention escalating chest pain and uncontrolled diabetes."

In [13]:
def retrieve_patient_history(patient_id, df):
    """
    Retrieves all clinical notes and diagnoses for a specific patient.
    In a production app, this would query a Vector Database (like FAISS or Pinecone).
    """
    patient_records = df[df['patient_id'] == patient_id].sort_values(by='date', ascending=False)

    if patient_records.empty:
        return "No records found for this patient."

    history_context = f"--- Clinical History for Patient {patient_id} ---\n"

    for index, row in patient_records.iterrows():
        history_context += f"Date: {row['date']} | Diagnosis Code: {row['dx_code']}\n"
        history_context += f"Clinical Note: {row['note']}\n"
        history_context += "-" * 40 + "\n"

    return history_context

# Let's test it on a patient (Replace H10009 with a valid ID from your dataset if needed)
sample_patient = df_merged['patient_id'].iloc[0]
patient_context = retrieve_patient_history(sample_patient, df_merged)

print(patient_context)

--- Clinical History for Patient H10000 ---
Date: 2024-03-15 | Diagnosis Code: E78
Clinical Note: Patient presents with headache for 1 days. History of diabetes. Vitals stable.
----------------------------------------



Step 5: Advanced Prompt Engineering for Clinical Explanations
What we are doing: We are designing a structured prompt that acts as a set of instructions for the LLM. It combines a "System Persona" (telling the AI to act like a clinical assistant), the patient's baseline structured data, and the context we retrieved from our RAG function. We will then simulate the LLM generating a summary.
Why: Doctors are busy and need high-signal, low-noise information. If we just hand the LLM raw database rows, it might hallucinate or provide irrelevant information. By strictly formatting the prompt, we force the model to provide a concise, factual, and actionable explanation of why the patient is flagged for risk.

In [14]:
# Function to construct the prompt
def build_clinical_prompt(patient_id, df_structured, patient_context):
    # Get the patient's structured baseline data
    patient_data = df_structured[df_structured['patient_id'] == patient_id].iloc[0]
    baseline_score = patient_data['structured_risk_score']
    age = patient_data['age']
    bmi = patient_data['bmi']

    # 1. System Prompt: Defines the persona and rules
    system_prompt = (
        "You are an expert AI clinical assistant. Your job is to help doctors understand "
        "patient risk. You will be provided with a patient's baseline risk score, demographics, "
        "and their recent clinical history. \n"
        "Rules:\n"
        "- Be concise, professional, and objective.\n"
        "- Summarize the key risk factors driving their current health status.\n"
        "- Do NOT diagnose the patient or prescribe medication.\n"
        "- Only use the information provided in the clinical context."
    )

    # 2. User Prompt: Injects the specific data for this patient
    user_prompt = f"""
    Patient ID: {patient_id}
    Age: {age} | BMI: {bmi}
    Baseline Structured Risk Score: {baseline_score} (Higher is worse)

    Clinical History Retrieved via RAG:
    {patient_context}

    Based on the above data, please provide a 3-4 sentence clinical summary explaining why this patient might be at risk and what the primary areas of concern are.
    """

    return system_prompt, user_prompt

# Function to simulate calling an LLM (like GPT-4, Claude, or Gemini)
def simulate_llm_call(system_prompt, user_prompt):
    """
    In a real production environment, you would use an API client here.
    Example:
    response = client.chat.completions.create(
        model="gpt-4",
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt}
        ]
    )
    return response.choices[0].message.content
    """

    print("🤖 Sending prompt to LLM... Generating response...\n")

    # Simulated intelligent response based on the prompt structure
    simulated_output = (
        "Clinical Summary:\n"
        "Patient exhibits an elevated baseline risk score driven by advanced age and high BMI. "
        "A review of the clinical history indicates recurrent episodes of severe back pain (M54) and "
        "anxiety (F41), alongside a documented history of GERD. The primary areas of concern are "
        "managing the chronic pain and addressing the ongoing anxiety, which may be exacerbating "
        "their physical symptoms. Close monitoring of their gastrointestinal health in relation to "
        "anxiety-induced stress is recommended."
    )
    return simulated_output

# --- Let's execute the pipeline for our sample patient ---

# 1. We define the patient we are looking at (using the sample from the previous step)
patient_to_review = sample_patient

# 2. We build the strict prompt
sys_prompt, usr_prompt = build_clinical_prompt(patient_to_review, df_merged, patient_context)

# 3. We review the prompt we are about to send (Good for debugging!)
print("--- CONSTRUCTED PROMPT ---")
print(usr_prompt)
print("-" * 40 + "\n")

# 4. We call the LLM
ai_explanation = simulate_llm_call(sys_prompt, usr_prompt)

# 5. Display the final result for the doctor
print("--- AI GENERATED RISK EXPLANATION ---")
print(ai_explanation)

--- CONSTRUCTED PROMPT ---

    Patient ID: H10000
    Age: 68 | BMI: 25.3
    Baseline Structured Risk Score: 2 (Higher is worse)
    
    Clinical History Retrieved via RAG:
    --- Clinical History for Patient H10000 ---
Date: 2024-03-15 | Diagnosis Code: E78
Clinical Note: Patient presents with headache for 1 days. History of diabetes. Vitals stable.
----------------------------------------

    
    Based on the above data, please provide a 3-4 sentence clinical summary explaining why this patient might be at risk and what the primary areas of concern are.
    
----------------------------------------

🤖 Sending prompt to LLM... Generating response...

--- AI GENERATED RISK EXPLANATION ---
Clinical Summary:
Patient exhibits an elevated baseline risk score driven by advanced age and high BMI. A review of the clinical history indicates recurrent episodes of severe back pain (M54) and anxiety (F41), alongside a documented history of GERD. The primary areas of concern are managing the